# Statistics and Experimentation for Data Engineers


*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## Why do averages lie and how do percentiles fix that

Averages are dragged by the tail while percentiles describe the shape users actually feel. A p50 of 14 ms with a p99 of 900 ms means one request in a hundred is unusable even though the mean looks acceptable.

```text
latencies = [10, 12, 13, 15, 20, 900]
mean = 161.7, p50 = 14, p95 = 900, p99 = 900
```

In [1]:
import statistics

def percentile(ordered: list[float], pct: float) -> float:
    # Nearest-rank definition: the smallest value at or above the pct-th rank.
    rank = max(1, int(-(-pct / 100 * len(ordered) // 1)))
    return ordered[rank - 1]

def summarize(values: list[float]) -> dict[str, float]:
    if not values:
        raise ValueError("empty sample")
    ordered = sorted(values)
    return {
        "n": len(ordered),
        "mean": statistics.fmean(ordered),
        "p50": percentile(ordered, 50),
        "p95": percentile(ordered, 95),
        "p99": percentile(ordered, 99),
        "max": ordered[-1],
    }

Interpretation: report p50 with p95 and p99, never the mean alone. Edge cases: percentiles are not additive, so p95 of a day is not the average of p95 per hour; with fewer than 100 samples p99 is the maximum; a bimodal distribution has a meaningless mean even when the tails look fine.

Mental model: the mean is the balance point of the histogram, so one heavy value moves it; the median is the value that splits the count in half, so it barely moves. Percentiles describe the shape at a chosen rank, which is why latency objectives are written on p95 and p99 rather than on the mean.

How it works: the nearest-rank percentile returns an observation that really occurred; an interpolated percentile (SQL `percentile_cont`, numpy) can return a value no user ever saw. With fewer than 100 points there is no p99 estimate, only the maximum.

Worked example: latencies `[10, 12, 13, 15, 20, 900]` have mean 161.7 and median 14, so the mean is 11.6 times the median and describes the tail, not the typical request.

Common mistake: quoting an average latency as if it were the p99 objective. Averages also hide bimodality: a 50/50 mix of 5 ms and 500 ms has a mean of 252 ms, and no request is anywhere near it.

## How do you summarize a distribution with descriptive statistics

Statement: before testing anything, describe center, spread, and shape. Standard deviation measures absolute spread; the coefficient of variation makes spread comparable across metrics with different units.

```text
values = [2, 4, 4, 4, 5, 5, 7, 9] -> mean 5.0, stdev 2.14, cv 0.43
```

In [2]:
import math
import statistics

def describe(values: list[float]) -> dict[str, float]:
    n = len(values)
    if n < 2:
        raise ValueError("need at least two observations")
    mean = statistics.fmean(values)
    stdev = statistics.stdev(values)
    centered = [(value - mean) / stdev for value in values]
    skew = n / ((n - 1) * (n - 2)) * sum(c**3 for c in centered) if n > 2 else 0.0
    return {
        "n": n,
        "mean": mean,
        "median": statistics.median(values),
        "stdev": stdev,
        "variance": statistics.variance(values),
        "cv": stdev / mean if mean else math.inf,
        "min": min(values),
        "max": max(values),
        "skew": skew,
    }

Interpretation: compare mean and median to see skew — `mean > median` means a right tail, so a small group of large values dominates. Edge cases: `stdev` uses the sample denominator `n - 1` while `pstdev` uses `n`, and mixing them changes the answer; a zero mean makes the coefficient of variation infinite, so use it only for ratio-scale metrics; `statistics.variance` needs at least two points.

Mental model: center, spread, and shape are three separate questions. Mean and median answer center, standard deviation and IQR answer spread, skew answers shape. Knowing one does not give you the others.

How it works: the sample standard deviation divides by `n - 1` because spread is measured around the sample's own mean; the population form divides by `n`. SQL's `stddev_samp` and `stddev_pop`, and numpy's `ddof=1` versus `ddof=0`, are the same distinction.

Worked example: `[2, 4, 4, 4, 5, 5, 7, 9]` has mean 5.0 and sample stdev 2.14, so the coefficient of variation is 0.43. The same stdev on a metric averaging 1,000 is a cv of 0.002.

Common mistake: comparing standard deviations across metrics with different units or scales. Convert to the coefficient of variation first.

## How do distributions and skew affect engineering decisions

Heavy-tailed metrics break the intuition built on normal distributions. Latency, revenue per user, and file sizes are right-skewed: most values are small, a few are huge, and the mean sits above the median.

```text
normal mean == median, thin tails, 3-sigma rule applies
log-normal mean > median, heavy right tail, 3-sigma rule fails
```

In [3]:
import math
import statistics

def compare_tail_models(values: list[float]) -> dict[str, float]:
    mean = statistics.fmean(values)
    median = statistics.median(values)
    stdev = statistics.stdev(values)
    beyond = sum(1 for value in values if abs(value - mean) > 3 * stdev) / len(values)
    logged = [math.log1p(value) for value in values]
    return {
        "mean_over_median": mean / median if median else math.inf,
        "share_beyond_3sigma": beyond,
        "logged_mean": statistics.fmean(logged),
        "logged_stdev": statistics.stdev(logged),
    }

Interpretation: when the mean-to-median ratio is well above 1 and the share beyond three sigma is far above the expected 0.3%, the metric is heavy-tailed. Aggregate on the log scale or on quantiles, and use medians for per-tenant comparisons. Edge cases: log of zero or negative values (`log1p` still fails on negatives); a heavy tail makes standard tests underpowered, so prefer the Mann-Whitney U test or a bootstrap; averaging ratios such as revenue per session is biased, so aggregate numerator and denominator separately.

Mental model: the normal distribution is symmetric with thin tails. Most operational data is right-skewed or log-normal, meaning the log of the value is roughly normal. Under right skew the mean sits above the median, and the gap grows with the tail weight.

How it works: for a log-normal the mean is `exp(mu + sigma^2 / 2)` and the median is `exp(mu)`, so `sigma` alone drives the mean-to-median ratio. The 68/95/99.7 rule applies on the log scale, not the raw scale.

Worked example: request latency with median 14 ms and mean 162 ms has a mean-to-median ratio of 11.6, a clear signal to model the log of latency.

Common mistake: running a t test or a 3-sigma rule on a heavy-tailed metric. The tail inflates the standard deviation, which hides the very outliers you want to find. Aggregate on quantiles or on the log scale.

## How do you sample data without bias

A sample is only useful if it is representative. Simple random sampling is the default; stratified sampling guarantees coverage of small segments; and cluster sampling is for when rows arrive in groups such as files.

```text
population: 1M users, 5% enterprise
bad sample: first 10k by user_id -> 0% enterprise (ids are created in cohort order)
good sample: stratified, 2k per segment
```

In [4]:
import random
from collections import defaultdict
from collections.abc import Iterable, Sequence
from typing import Any, TypeVar

T = TypeVar("T")

def simple_random(rows: Sequence[T], size: int, seed: int = 7) -> list[T]:
    return random.Random(seed).sample(rows, min(size, len(rows)))

def stratified(rows: Iterable[dict[str, Any]], stratum: str, per_stratum: int, seed: int = 7) -> list[dict[str, Any]]:
    buckets: dict[Any, list[dict[str, Any]]] = defaultdict(list)
    for row in rows:
        buckets[row[stratum]].append(row)
    rng = random.Random(seed)
    return [row for bucket in buckets.values() for row in rng.sample(bucket, min(per_stratum, len(bucket)))]

Interpretation: fix and record the seed and the salt so the sample is reproducible and backfills do not reshuffle. Edge cases: sampling the first N rows of a time-ordered table biases toward old records; filtering the population after sampling breaks the ratio; a hash sample is consistent across jobs but is not exactly uniform at small sizes; a segment that appears in 0.1% of rows needs stratification or it disappears.

Mental model: sampling moves the randomness from the analysis into the selection. A sample is representative when every population unit has a known, non-zero chance of selection and that chance does not correlate with the metric.

How it works: simple random sampling draws independently, stratified sampling draws within each segment, cluster sampling draws whole groups, and systematic sampling draws every k-th row. Systematic sampling fails when the data has a period matching k.

Worked example: taking the first 10k `user_id` values from a table ordered by creation date returns zero enterprise users when enterprises onboarded late; stratifying at 2k per segment keeps their share.

Common mistake: filtering after sampling. Removing rows changes the denominator and the ratio, so filter the population first and sample second.

## How do you interpret a confidence interval

A 95% confidence interval is a range that would contain the true parameter in 95% of repeated samples. It is not a 95% probability that the parameter is in this particular interval, which is the most common interview mistake.

```text
n = 100, mean = 20, stdev = 10 -> 95% CI = [18.04, 21.96]
```

In [5]:
import math
import statistics

def ci_mean(values: list[float], z: float = 1.959964) -> tuple[float, float]:
    n = len(values)
    if n < 2:
        raise ValueError("need at least two observations")
    mean = statistics.fmean(values)
    standard_error = statistics.stdev(values) / math.sqrt(n)
    return mean - z * standard_error, mean + z * standard_error

Interpretation: the interval narrows with the square root of n, so quadrupling the sample halves the width. Overlapping intervals are not a valid significance test, because each interval carries its own error; run the two-sample test instead. Edge cases: the normal approximation for a proportion is unreliable when `n * p < 5`, so use Wilson or Clopper-Pearson; the z multiplier is wrong for small n, where a t distribution is needed; clustering (many rows per user) inflates the effective variance, so compute the interval on per-user aggregates.

Mental model: the true mean is fixed and the interval is random. Across repeated samples, 95% of intervals built this way cover the parameter; any single interval either does or does not. The confidence level describes the procedure, not this one interval.

How it works: the interval is `estimate +/- z * standard error`, and the standard error is `sd / sqrt(n)`. Halving the width needs four times the data. The multiplier is 1.96 at 95% for large samples and larger for small ones, where the t distribution applies.

Worked example: n = 100, mean 20, sd 10 gives a standard error of 1.0 and a 95% CI of [18.04, 21.96].

Common mistake: saying there is a 95% probability the true mean lies in this interval. That is a Bayesian credible-interval statement and needs a prior. A second mistake: treating a narrow interval as proof of correctness, when a biased sample gives a tight interval around the wrong number.

## How do you run a t test and a proportion test

The t test compares two means; the two-proportion z test compares two conversion rates. Both ask the same question: is the observed difference large relative to the noise?

```text
control:   mean 20.0 (n=500), treatment: mean 21.5 (n=500)
proportion: 10/100 vs 18/100 -> z = 1.63, p = 0.104
```

In [6]:
try:
    import math

    from scipy import stats

    def welch_t_test(control: list[float], treatment: list[float]) -> dict[str, float]:
        # Welch does not assume equal variances, which is the safe default.
        result = stats.ttest_ind(control, treatment, equal_var=False)
        return {"t": float(result.statistic), "p_value": float(result.pvalue)}

    def two_proportion_z(success_a: int, n_a: int, success_b: int, n_b: int) -> dict[str, float]:
        p_a, p_b = success_a / n_a, success_b / n_b
        pooled = (success_a + success_b) / (n_a + n_b)
        standard_error = math.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
        if standard_error == 0:
            return {"z": 0.0, "p_value": 1.0}
        z = (p_b - p_a) / standard_error
        p_value = 2 * (1 - stats.norm.cdf(abs(z)))
        return {"z": z, "p_value": p_value, "absolute_lift": p_b - p_a}
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - scipy is not installed; install it and re-run this cell.


Interpretation: report the effect size with the p-value; a tiny but real lift is still not worth shipping. Edge cases: the t test assumes independent observations, so repeated rows per user understate the variance; the proportion test needs at least about 5 successes and 5 failures per arm; one-sided tests double the false-positive risk if the direction was chosen after looking at the data; use the Mann-Whitney U test when the metric is heavy-tailed.

Mental model: both tests divide an observed difference by its standard error. The t test uses the spread of the two samples; the proportion test uses the binomial variance `p(1-p)`.

How it works: Welch's t test does not assume equal variances, so it is the safe default; the two-proportion z test pools the rates under the null and reports an absolute lift. Both test a difference of exactly zero, which is rarely the decision question.

In [7]:
import math

def two_sided_p_from_z(z: float) -> float:
    # Normal approximation to the two-sided tail; exact enough for large samples.
    return math.erfc(abs(z) / math.sqrt(2))

def two_proportion_z(success_a: int, n_a: int, success_b: int, n_b: int) -> dict[str, float]:
    p_a, p_b = success_a / n_a, success_b / n_b
    pooled = (success_a + success_b) / (n_a + n_b)
    se = math.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    if se == 0:
        return {"z": 0.0, "p_value": 1.0, "absolute_lift": 0.0}
    z = (p_b - p_a) / se
    return {"z": z, "p_value": two_sided_p_from_z(z), "absolute_lift": p_b - p_a}

result = two_proportion_z(10, 100, 18, 100)
print(f"z = {result['z']:.3f}, p = {result['p_value']:.3f}, lift = {result['absolute_lift']:.2f}")
assert round(result["z"], 3) == 1.630
assert round(result["p_value"], 3) == 0.103

z = 1.630, p = 0.103, lift = 0.08


Worked example: 10/100 against 18/100 pools to 0.14, giving a standard error of 0.049, z = 1.63, and a two-sided p of about 0.10.

Common mistake: reading a non-significant result as proof of no effect. An underpowered test is fully consistent with a real lift, so report the confidence interval, which shows the range of effects the data are compatible with.

## How do you use a chi square test on categorical data

The chi-square test compares observed counts to the counts expected under independence. It answers questions like "is the conversion rate different across channels" or "are missing values evenly distributed across tenants".

```text
             converted  not converted
control            200            800
treatment          250            750
chi2 = 7.17, dof = 1, p = 0.0074, Cramer's V = 0.060
```

In [8]:
try:
    from scipy.stats import chi2_contingency

    def chi_square(table: list[list[int]]) -> dict[str, float]:
        chi2, p_value, dof, expected = chi2_contingency(table, correction=False)
        n = sum(sum(row) for row in table)
        cramers_v = (chi2 / (n * (min(len(table), len(table[0])) - 1))) ** 0.5
        return {"chi2": float(chi2), "p_value": float(p_value), "dof": dof, "cramers_v": float(cramers_v)}

    def expected_counts(table: list[list[int]]) -> list[list[float]]:
        _chi2, _p, _dof, expected = chi2_contingency(table, correction=False)
        return [[round(value, 2) for value in row] for row in expected]
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - scipy is not installed; install it and re-run this cell.


Interpretation: Cramer's V gives effect size, since a large table makes chi-square large even for trivial associations. Edge cases: any expected cell below 5 invalidates the approximation, so merge categories or use Fisher's exact test; the test needs counts, not percentages; a significant result on a 10x10 table does not say which cell drove it, so inspect standardized residuals; repeated measures violate independence.

Mental model: chi-square measures how far observed counts sit from the counts expected if the two variables were independent. The expected count in a cell is `row_total * column_total / grand_total`.

In [9]:
import math

def chi_square_2x2(table: list[list[int]]) -> dict[str, float]:
    rows = [sum(row) for row in table]
    cols = [sum(column) for column in zip(*table)]
    total = sum(rows)
    chi2 = 0.0
    for i in range(2):
        for j in range(2):
            expected = rows[i] * cols[j] / total
            chi2 += (table[i][j] - expected) ** 2 / expected
    # For one degree of freedom, p = erfc(sqrt(chi2 / 2)).
    p_value = math.erfc(math.sqrt(chi2 / 2))
    cramers_v = math.sqrt(chi2 / total)
    return {"chi2": chi2, "p_value": p_value, "cramers_v": cramers_v}

result = chi_square_2x2([[200, 800], [250, 750]])
print(f"chi2 = {result['chi2']:.2f}, p = {result['p_value']:.4f}, V = {result['cramers_v']:.3f}")
assert round(result["chi2"], 2) == 7.17
assert round(result["p_value"], 4) == 0.0074

chi2 = 7.17, p = 0.0074, V = 0.060


How it works: the statistic is the sum of `(observed - expected)^2 / expected`, with `(rows - 1) * (columns - 1)` degrees of freedom. It tests any association, not a direction and not one particular cell.

Worked example: 200/800 against 250/750 has expected cells of 225 and 775, chi2 = 7.17 on one degree of freedom, p = 0.0074, and Cramer's V = 0.060: statistically clear, practically tiny.

Common mistake: reading chi-square as a comparison of percentages. It compares counts, so a huge table makes a trivial association significant. Inspect standardized residuals to see which cell drove the result, and merge categories when any expected count is below 5.

## What do p values actually mean and how do multiple comparisons break them

A p-value is the probability of data at least this extreme if the null hypothesis were true. It is not the probability that the null is true, and it is not the probability that a result replicates.

```text
20 independent A/A tests at alpha = 0.05 -> expect ~1 false positive
bonferroni: alpha / 20 = 0.0025 per test
```

In [10]:
def benjamini_hochberg(p_values: list[float], alpha: float = 0.05) -> dict[str, list]:
    # Controls the false discovery rate, which is usually the right target.
    m = len(p_values)
    ranked = sorted(enumerate(p_values), key=lambda item: item[1])
    threshold_rank = 0
    for rank, (_index, p_value) in enumerate(ranked, start=1):
        if p_value <= rank / m * alpha:
            threshold_rank = rank
    rejected = [index for index, _p in ranked[:threshold_rank]]
    return {"rejected": rejected, "cutoff": ranked[threshold_rank - 1][1] if threshold_rank else None}

def bonferroni(p_values: list[float], alpha: float = 0.05) -> list[int]:
    cutoff = alpha / len(p_values)
    return [index for index, p_value in enumerate(p_values) if p_value <= cutoff]

Interpretation: with many metrics, always correct for multiplicity; Bonferroni controls the chance of any false positive and is conservative, while Benjamini-Hochberg controls the share of false discoveries and is the better default for dashboards. Edge cases: correlated metrics make the correction overly conservative, so a permutation test that preserves correlation is more accurate; p-values depend on sample size, so a huge dataset makes trivial differences significant; a p-value with no effect size and no confidence interval is incomplete.

Mental model: a p-value is `P(data at least this extreme | null hypothesis)`. It is a statement about data given a hypothesis, not about a hypothesis given data.

How it works: under the null a p-value is uniform on [0, 1], so at alpha 0.05 one test in twenty rejects by construction. Bonferroni divides alpha by the number of tests. Benjamini-Hochberg sorts the p-values and rejects the largest rank k where `p_(k) <= k/m * alpha`, controlling the expected share of false discoveries.

Worked example: twenty independent A/A tests at alpha 0.05 expect about one false positive; Bonferroni sets the per-test cutoff to 0.0025.

Common mistake: saying there is a 5% probability the null is true, or that p = 0.03 means the result is 97% likely to replicate. Neither follows; replication depends on power and on the prior odds that the effect is real.

## How do you calculate statistical power and sample size

Power is the probability of detecting a real effect of a given size. Sample size, effect size, variance, and alpha trade off against each other; fixing three determines the fourth.

```text
baseline 10%, target 12%, alpha 0.05, power 0.80 -> ~3,842 per arm
baseline 10%, target 11% -> ~14,750 per arm
```

In [11]:
import math

Z_ALPHA = 1.959964  # two-sided alpha = 0.05
Z_POWER = 0.8416212  # power = 0.80

def sample_size_proportion(p1: float, p2: float, alpha: float = 0.05, power: float = 0.8) -> int:
    z_alpha = 1.959964 if abs(alpha - 0.05) < 1e-9 else 2.5758293
    z_power = 0.8416212 if abs(power - 0.8) < 1e-9 else 1.2815516
    pooled = (p1 + p2) / 2
    numerator = (z_alpha * math.sqrt(2 * pooled * (1 - pooled)) + z_power * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return math.ceil(numerator / (p2 - p1) ** 2)

Interpretation: halving the detectable effect quadruples the sample, which is why small improvements need long experiments or a larger randomization unit. Edge cases: the formula assumes independent Bernoulli observations and understates n for ratios or clustered data; never compute power after seeing the result to justify a non-significant test (observed power is a function of the p-value); multiple metrics and repeated looks each need their own correction.

Mental model: power is the probability of detecting an effect of a stated size when it exists. Alpha is the tolerated false-positive rate, beta is the tolerated miss rate, and power is 1 - beta.

How it works: required sample size grows with variance, grows with the power you demand, and falls with the square of the effect size. For two proportions the standard formula is roughly `16 * p(1-p) / delta^2` per arm at alpha 0.05 and 80% power, before any correction.

Worked example: baseline 10% and target 12% need about 3,841 per arm; a target of 11% needs about 14,751, roughly four times more for half the effect.

Common mistake: computing observed power after a non-significant result. Observed power is a monotone function of the p-value and adds no information. Use the minimum detectable effect from the design instead.

## How do you design an A B test for a data platform

An experiment is only valid if assignment is random, stable, and independent of the metric. The randomization unit must match the unit of analysis: users for retention, sessions for click-through, clusters when users can leak between variants.

```text
unit: user_id; variants: control/treatment; hash(user_id, salt) -> bucket
```

In [12]:
import hashlib

def assign(unit_id: str, experiment: str, buckets: int = 100, salt: str = "v1") -> int:
    digest = hashlib.sha256(f"{experiment}:{salt}:{unit_id}".encode()).hexdigest()
    return int(digest[:8], 16) % buckets

def variant(unit_id: str, experiment: str, traffic_split: float = 0.5) -> str:
    bucket = assign(unit_id, experiment)
    return "treatment" if bucket < traffic_split * 100 else "control"

Interpretation: specify the primary metric, the guardrails, the randomization unit, the minimum detectable effect, and the analysis window before launch; adding a metric after launch invalidates the p-value. Edge cases: the same user touching two variants contaminates the comparison; changing the salt mid-experiment reshuffles users and destroys the sample; network effects between users break the independence assumption; ramp-ups that expose heavy users first create a novelty-driven sample bias.

Mental model: an experiment compares counterfactuals. Randomization makes the two groups exchangeable on average, so the difference in outcomes estimates the effect, provided assignment is random, stable, and independent of the metric.

How it works: hash the randomization unit with a fixed salt into buckets, keep that unit constant for the whole experiment, and analyze at the unit level so a user contributes one row rather than one row per session.

Worked example: `hash(user_id, 'exp_v1') % 100` assigns buckets 0-49 to control and 50-99 to treatment, and the same user always lands in the same arm.

Common mistake: randomizing by session while analyzing by user, or changing the salt mid-flight. Both break exchangeability and surface later as sample ratio mismatch. Ramp-ups also bias the early sample toward heavy users.

## What is sample ratio mismatch and how do you detect it

Sample ratio mismatch (SRM) means the observed traffic split differs from the intended split by more than chance allows. It is the single most useful health check because it is evidence that assignment or logging is broken.

```text
intended 50/50, observed control = 49,100, treatment = 50,900, n = 100,000
chi2 = 32.4, p = 1.2e-08 -> SRM: do not trust the result
```

In [13]:
try:
    from scipy.stats import chisquare

    def srm_check(counts: dict[str, int], ratios: dict[str, float], alpha: float = 0.001) -> dict[str, float]:
        total = sum(counts.values())
        observed = [counts[name] for name in counts]
        expected = [ratios[name] * total for name in counts]
        statistic, p_value = chisquare(observed, expected)
        return {"total": total, "chi2": float(statistic), "p_value": float(p_value), "srm": float(p_value) < alpha}
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - scipy is not installed; install it and re-run this cell.


Interpretation: run the check overall and per platform, device, and country, because a global 50/50 can hide a broken split inside one segment. Edge cases: a strict alpha such as 0.001 avoids false alarms from routine checks; a real SRM means the experiment is invalid, not that a correction is needed; a small SRM can be caused by bot traffic or by a logging pipeline that drops one variant; bot filtering and assignment must use the same key or the counts never match.

Mental model: SRM is a data-integrity alarm, not a statistical result. A 50/50 design that observes 49,100 against 50,900 is far outside chance and means assignment or logging is broken.

In [14]:
import math

def srm_check(observed: list[int], expected: list[float], alpha: float = 0.001) -> dict[str, float]:
    chi2 = sum((o - e) ** 2 / e for o, e in zip(observed, expected))
    # One degree of freedom for a two-arm split.
    p_value = math.erfc(math.sqrt(chi2 / 2))
    return {"chi2": chi2, "p_value": p_value, "srm": float(p_value) < alpha}

result = srm_check([49_100, 50_900], [50_000, 50_000])
print(f"chi2 = {result['chi2']:.1f}, p = {result['p_value']:.2e}, srm = {result['srm']}")
assert round(result["chi2"], 1) == 32.4
assert result["srm"] is True

chi2 = 32.4, p = 1.25e-08, srm = True


How it works: a chi-square goodness-of-fit test compares observed to expected counts, and a strict alpha such as 0.001 is normal because the check runs constantly and across many segments.

Worked example: with 100,000 users expected 50,000/50,000, observing 49,100/50,900 gives chi2 = 32.4 and p about 1.3e-08, so the readout is invalid.

Common mistake: correcting for SRM by reweighting the arms. The mismatch usually means the arms are not comparable at all, so find the assignment or logging bug and rerun.

## What are guardrail metrics and novelty effects

Guardrails are metrics that must not degrade even if the primary metric improves: latency, error rate, refunds, unsubscribe rate, data freshness. Novelty effects are temporary behavior changes caused by the newness of the treatment rather than the treatment itself.

```text
primary: conversion rate +1.5%
guardrail: p99 latency +40% -> do not ship even though the primary metric won
```

In [15]:
def guardrail_breach(control: list[float], treatment: list[float], tolerance: float = 0.0) -> dict[str, float]:
    mean_control = sum(control) / len(control)
    mean_treatment = sum(treatment) / len(treatment)
    relative = (mean_treatment - mean_control) / mean_control if mean_control else 0.0
    return {"control_mean": mean_control, "treatment_mean": mean_treatment, "relative_change": relative, "breached": relative > tolerance}

Interpretation: pre-register guardrails with thresholds and hold the launch decision on them; a positive early lift that decays over two weeks is usually novelty, and the analysis should use the mature window rather than the whole run. Edge cases: guardrails with low traffic need longer windows before they can detect a regression; multiple guardrails need multiplicity control; a seasonal week can mimic a novelty decay; stopping early on a guardrail is itself a peeking problem, so use sequential methods.

Mental model: the primary metric answers whether it worked; guardrails answer whether it broke something else. A launch needs the first to win and the guardrails not to lose.

How it works: pre-register each guardrail with a direction and a non-inferiority threshold, judge the primary on the pre-registered window, and control multiplicity across the guardrails.

Worked example: a +1.5% conversion lift with a +40% p99 latency regression is not a win; the regression is a guardrail breach and blocks the launch.

Common mistake: stopping the moment the primary metric looks good. That is peeking, it inflates false positives, and it hides a slow guardrail regression that only appears after novelty wears off.

## What is sequential testing and why does peeking inflate false positives

Checking a test every day and stopping when p < 0.05 turns a 5% false-positive rate into a much larger one. Sequential methods hold the overall error rate while allowing valid interim looks.

```text
fixed-horizon alpha = 0.05; ten daily looks with naive peeking -> far above 0.05
use an alpha-spending boundary or a sequential test instead
```

In [16]:
try:
    import numpy as np
    from scipy.stats import chi2_contingency, norm

    def peek_false_positive_rate(looks: int = 10, per_look: int = 1000, baseline: float = 0.1, trials: int = 2000, seed: int = 0) -> float:
        rng = np.random.default_rng(seed)
        false_positives = 0
        for _ in range(trials):
            control = rng.binomial(1, baseline, looks * per_look).cumsum()
            treatment = rng.binomial(1, baseline, looks * per_look).cumsum()
            for look in range(1, looks + 1):
                k = look * per_look
                table = [[control[k - 1], k - control[k - 1]], [treatment[k - 1], k - treatment[k - 1]]]
                _chi2, p_value, _dof, _expected = chi2_contingency(table, correction=False)
                if p_value < 0.05:
                    false_positives += 1
                    break
        return false_positives / trials

    def alpha_spending_thresholds(looks: int, alpha: float = 0.05) -> list[float]:
        # Pocock-style constant boundary approximation for illustration.
        z = norm.ppf(1 - alpha / (2 * looks))
        return [round(float(z), 3)] * looks
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - numpy is not installed; install it and re-run this cell.


Interpretation: decide the look schedule before launch and use a boundary that accounts for it; a sequential test spends alpha across looks so the final decision is still valid. Edge cases: the simulation above runs an A/A test, so any rejection is by construction a false positive; unequal look sizes and unequal traffic need the boundary recomputed; peeking at many metrics multiplies the problem; a Bayes factor or an always-valid confidence sequence is an alternative that needs no fixed schedule.

Mental model: every look at accumulating data is another chance to cross the threshold. A fixed-horizon alpha assumes exactly one look, so daily peeking turns a 5% error rate into a much larger one.

In [17]:
import math
import random

def two_sided_p_from_z(z: float) -> float:
    return math.erfc(abs(z) / math.sqrt(2))

def peek_false_positive_rate(looks: int = 10, per_look: int = 200, baseline: float = 0.1,
                             trials: int = 400, seed: int = 0) -> float:
    rng = random.Random(seed)
    false_positives = 0
    for _ in range(trials):
        control_n = treatment_n = control_s = treatment_s = 0
        for _look in range(looks):
            for _ in range(per_look):
                control_s += rng.random() < baseline
                treatment_s += rng.random() < baseline
            control_n += per_look
            treatment_n += per_look
            p_c, p_t = control_s / control_n, treatment_s / treatment_n
            pooled = (control_s + treatment_s) / (control_n + treatment_n)
            se = math.sqrt(pooled * (1 - pooled) * (1 / control_n + 1 / treatment_n))
            if se and two_sided_p_from_z((p_t - p_c) / se) < 0.05:
                false_positives += 1
                break
    return false_positives / trials

# The simulation is an A/A test, so every rejection is a false positive by construction.
rate = peek_false_positive_rate(seed=0)
print(f"family-wise false-positive rate with 10 looks = {rate:.3f}")
assert rate > 0.10  # well above the nominal 0.05

family-wise false-positive rate with 10 looks = 0.212


How it works: alpha spending divides the alpha budget across looks and sets a wider boundary early; an always-valid confidence sequence keeps coverage at every sample size and needs no schedule.

Common mistake: choosing the look schedule after seeing the data, or peeking at many metrics and reporting the first that crosses. Pre-register both the schedule and the metric list.

## Bayesian vs frequentist approaches to experiments

Frequentist inference fixes the parameter and treats the data as random; Bayesian inference fixes the data and treats the parameter as uncertain. Bayesian results answer the question decision-makers actually ask: "what is the probability that B is better?"

```text
A: 100/1000, B: 130/1000 -> P(B > A) ~ 0.98, expected loss ~ 0.006
```

In [18]:
try:
    import numpy as np

    def posterior_samples(successes: int, n: int, draws: int = 200_000, seed: int = 0) -> np.ndarray:
        # Beta(1, 1) prior keeps the model equivalent to add-one smoothing.
        rng = np.random.default_rng(seed)
        return rng.beta(successes + 1, n - successes + 1, draws)

    def prob_b_beats_a(success_a: int, n_a: int, success_b: int, n_b: int, draws: int = 200_000, seed: int = 0) -> float:
        a = posterior_samples(success_a, n_a, draws, seed)
        b = posterior_samples(success_b, n_b, draws, seed + 1)
        return float((b > a).mean())
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - numpy is not installed; install it and re-run this cell.


Interpretation: the posterior probability is easy to explain but depends on the prior and is not a guarantee; the frequentist p-value has a precise long-run meaning but is easy to misread. Edge cases: a weakly informative prior still moves the answer when traffic is low; Bayesian credible intervals are not confidence intervals, so do not swap the labels; optional stopping is more defensible under a Bayesian model but the expected loss still needs a decision threshold; simulation cost grows with the number of metrics.

Mental model: frequentists treat the parameter as fixed and the data as random; Bayesians treat the data as fixed and the parameter as uncertain, updating a prior into a posterior.

In [19]:
import random

def prob_b_beats_a(success_a: int, n_a: int, success_b: int, n_b: int,
                   draws: int = 20_000, seed: int = 0) -> float:
    # Beta(1, 1) prior with binomial data gives a Beta(1 + s, 1 + f) posterior.
    rng = random.Random(seed)
    wins = 0
    for _ in range(draws):
        a = rng.betavariate(success_a + 1, n_a - success_a + 1)
        b = rng.betavariate(success_b + 1, n_b - success_b + 1)
        wins += b > a
    return wins / draws

probability = prob_b_beats_a(100, 1000, 130, 1000)
print(f"P(B > A) = {probability:.3f}")
assert probability > 0.95

P(B > A) = 0.983


How it works: with a Beta prior and binomial data the posterior is `Beta(a + successes, b + failures)`. `P(B > A)` is the share of posterior draws where B exceeds A, and the expected loss is the average amount lost by picking the better arm.

Worked example: 100/1000 against 130/1000 with a Beta(1,1) prior gives `P(B > A)` near 0.98, which maps directly to a ship decision.

Common mistake: calling a credible interval a confidence interval, or the reverse. A flat prior is still a choice and it moves the answer when traffic is low.

## How do you distinguish correlation from causation and handle confounding

Correlation is a measure of linear association; causation needs a mechanism plus a design that removes confounding. A confounder changes both the treatment and the outcome, and it creates a correlation where none is causal.

```text
ice cream sales correlate with drownings; the confounder is temperature
revenue correlates with support tickets; the confounder is customer size
```

In [20]:
try:
    import numpy as np

    def correlation(x: list[float], y: list[float]) -> float:
        return float(np.corrcoef(x, y)[0, 1])

    def partial_correlation(x: list[float], y: list[float], z: list[float]) -> float:
        # Correlate x and y after linearly removing the confounder z from each.
        design = np.column_stack([np.ones(len(z)), z])
        residual_x = np.asarray(x) - design @ np.linalg.lstsq(design, np.asarray(x), rcond=None)[0]
        residual_y = np.asarray(y) - design @ np.linalg.lstsq(design, np.asarray(y), rcond=None)[0]
        return float(np.corrcoef(residual_x, residual_y)[0, 1])
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - numpy is not installed; install it and re-run this cell.


Interpretation: report correlation with a scatter plot and the confounders you adjusted for; a large drop from correlation to partial correlation is evidence that z explains the association. Edge cases: adjusting for a mediator removes a real causal effect, and adjusting for a collider creates a fake one; Pearson correlation misses non-linear relationships, so check Spearman and the plot; regression controls only for measured confounders, so unmeasured ones still bias the estimate; the only clean proof of causation is a randomized experiment or a credible quasi-experimental design.

Mental model: correlation measures linear association; causation needs a mechanism plus a design that closes backdoor paths. A confounder is a common cause of both treatment and outcome.

How it works: partial correlation or regression removes the linear part of the confounder from both variables, so the remaining association is measured at a fixed level of the confounder. That works only for measured confounders; unmeasured ones still bias the estimate.

Worked example: ice cream sales correlate with drownings, but temperature drives both, and conditioning on it collapses the association. Revenue correlates with support tickets, but customer size drives both.

Common mistake: adjusting for a mediator, which removes a real effect, or for a collider, which creates a fake one. Decide from the causal graph which variables belong in the adjustment set before fitting anything.

## What is Simpson's paradox

An association that holds in every subgroup can reverse when the subgroups are combined, because the groups differ in size and in baseline rate. It is a warning that aggregated metrics are not automatically safe.

```text
group A: control 1/10, treatment 2/10   (treatment better)
group B: control 50/100, treatment 40/100 (treatment worse)
aggregate: control 51/110, treatment 42/110 -> treatment appears better
```

In [21]:
def simpson_check(groups: list[dict]) -> dict[str, float | int]:
    treatment_n = sum(group["treatment_n"] for group in groups)
    treatment_success = sum(group["treatment_success"] for group in groups)
    control_n = sum(group["control_n"] for group in groups)
    control_success = sum(group["control_success"] for group in groups)
    aggregate = treatment_success / treatment_n - control_success / control_n
    total = treatment_n + control_n
    weighted = sum(
        (group["treatment_n"] + group["control_n"]) / total
        * (group["treatment_success"] / group["treatment_n"] - group["control_success"] / group["control_n"])
        for group in groups
    )
    signs_differ = (aggregate > 0) != (weighted > 0)
    return {"aggregate_lift": aggregate, "stratum_weighted_lift": weighted, "reversal": int(signs_differ)}

Interpretation: when exposure is uneven across strata, report the stratum-weighted or stratified estimate and state the weighting scheme. Edge cases: the paradox is not a statistical error but a real feature of unequal group sizes; the correct weight depends on the target population, so weighting by traffic answers a different question than weighting by users; a change in the mix of traffic over time can produce a false trend; a confounded experiment that lands unevenly across segments produces the same pattern.

Mental model: a pooled comparison weights each stratum by its traffic, and the traffic mix can differ from the effect. Simpson's paradox is the case where that weighting reverses the sign of the effect.

How it works: compute the effect within each stratum, then combine with fixed weights from the target population. The stratified estimate answers a different question from the pooled one, and both are correct for their own weighting.

Worked example: group A shows treatment better and group B shows treatment worse, yet the pooled numbers favor treatment because the groups differ in size and baseline.

Common mistake: reporting the pooled number when exposure is uneven across strata. Report the stratified estimate and state the weighting. An experiment that lands unevenly across segments produces the same pattern.

## How do you detect anomalies with z score IQR and MAD

Anomaly detection compares a value to what is normal for its context. The z-score assumes a roughly normal distribution, the IQR is robust to mild outliers, and MAD is the most robust to extreme ones.

```text
[10, 11, 12, 12, 13, 900] -> z-score flags 900; IQR flags 900; MAD flags 900
[10, 11, 12, 12, 13, 14] -> nothing flagged
```

In [22]:
import statistics

def zscore_flags(values: list[float], threshold: float = 3.0) -> list[int]:
    mean = statistics.fmean(values)
    stdev = statistics.stdev(values)
    if stdev == 0:
        return []
    return [index for index, value in enumerate(values) if abs(value - mean) / stdev > threshold]

def iqr_flags(values: list[float], k: float = 1.5) -> list[int]:
    quantiles = statistics.quantiles(values, n=4)
    q1, q3 = quantiles[0], quantiles[2]
    low, high = q1 - k * (q3 - q1), q3 + k * (q3 - q1)
    return [index for index, value in enumerate(values) if value < low or value > high]

def mad_flags(values: list[float], threshold: float = 3.5) -> list[int]:
    median = statistics.median(values)
    mad = statistics.median([abs(value - median) for value in values]) or 1e-9
    return [index for index, value in enumerate(values) if 0.6745 * abs(value - median) / mad > threshold]

Interpretation: MAD is the right default for metrics with a heavy tail because a single extreme value barely moves the median and the MAD. Edge cases: all three methods fail on a seasonal metric unless the value is compared to the same season; a slow drift produces no individual outlier, so pair point detection with a rolling baseline; `statistics.quantiles` needs enough points for a meaningful Q1 and Q3; a constant series has a zero MAD, hence the epsilon; alert thresholds need a minimum effect size as well as a statistical cutoff.

Mental model: a z-score standardizes by the mean and standard deviation; the IQR uses the middle half of the data; MAD uses the median absolute deviation. Each answers how far a point is, with a different definition of normal.

How it works: the robust z-score is `0.6745 * (x - median) / MAD`, scaled to match the standard z under a normal distribution; the IQR rule flags points beyond `Q1 - 1.5 * IQR` or `Q3 + 1.5 * IQR`.

Worked example: for `[10, 11, 12, 12, 13, 900]` the mean and standard deviation are dragged by 900, so the plain z-score of the outlier is only about 2.0 and misses it, while MAD gives a robust z above 500 and flags it instantly.

Common mistake: alerting on a fixed threshold with no minimum effect size, and computing the baseline over a window that already contains the outlier. Use a robust baseline and require both a statistical flag and a business-relevant magnitude.

## How do you decompose a time series into trend and seasonality

Decomposition splits a series into trend, seasonal, and residual components. The additive form fits constant seasonal amplitude; the multiplicative form fits amplitude that grows with the level, and it becomes additive after a log transform.

```text
values by week: [10, 12, 15, 11, 13, 16, 12, 14, 17], period = 3
trend = centered moving average; seasonal = mean detrended value per phase
```

In [23]:
def centered_moving_average(values: list[float], period: int) -> list[float | None]:
    half = period // 2
    trend: list[float | None] = [None] * len(values)
    for index in range(half, len(values) - half):
        window = values[index - half: index + half + 1]
        trend[index] = sum(window) / len(window)
    return trend

def seasonal_indices(values: list[float], period: int) -> list[float]:
    trend = centered_moving_average(values, period)
    buckets: list[list[float]] = [[] for _ in range(period)]
    for index, value in enumerate(values):
        if trend[index] is not None:
            buckets[index % period].append(value - trend[index])
    means = [sum(bucket) / len(bucket) if bucket else 0.0 for bucket in buckets]
    overall = sum(means) / period
    return [mean - overall for mean in means]

Interpretation: read the residual for anomalies, the seasonal index for capacity planning, and the trend for growth; a strong seasonal index means year-over-year comparisons, not week-over-week. Edge cases: a period that does not divide the series leaves the ends without a trend estimate; holidays and promotions must be handled as events, not as seasonality; an additive model on a multiplicative series leaves structure in the residuals; a real decomposition tool such as STL handles changing seasonality, which this simple version does not.

Mental model: an observation is trend times seasonal times residual (multiplicative) or trend plus seasonal plus residual (additive). The residual is what remains after the predictable parts are removed.

How it works: a centered moving average estimates the trend, the average detrended value per phase gives the seasonal index, and the remainder is the residual. Multiplicative seasonality becomes additive after a log transform.

Worked example: weekly values `[10, 12, 15, 11, 13, 16, 12, 14, 17]` with period 3 show a clear upward trend plus a repeating within-period shape.

Common mistake: treating a holiday or a one-off promotion as seasonality, and comparing week over week when the seasonal index is large. Compare year over year instead.

## What are stationarity and moving averages

A stationary series has a constant mean and variance over time. Most business series are not stationary: they trend, they have weekly seasonality, and their variance grows with volume, which breaks tests that assume stationarity.

```text
[1, 2, 3, 4, 5, 6] -> non-stationary (mean drifts)
[3, 2, 4, 3, 3, 4] -> roughly stationary
```

In [24]:
def moving_average(values: list[float], window: int) -> list[float | None]:
    out: list[float | None] = []
    total = 0.0
    for index, value in enumerate(values):
        total += value
        if index >= window:
            total -= values[index - window]
        out.append(total / window if index >= window - 1 else None)
    return out

def stationarity_check(values: list[float], parts: int = 4) -> dict[str, float]:
    # A quick diagnostic; a real ADF or KPSS test is the formal answer.
    chunk = max(1, len(values) // parts)
    means = [sum(values[i * chunk:(i + 1) * chunk]) / len(values[i * chunk:(i + 1) * chunk]) for i in range(parts)]
    variances = []
    for i in range(parts):
        piece = values[i * chunk:(i + 1) * chunk]
        mean = sum(piece) / len(piece)
        variances.append(sum((value - mean) ** 2 for value in piece) / max(1, len(piece) - 1))
    return {"mean_drift": max(means) - min(means), "variance_ratio": max(variances) / (min(variances) or 1e-9)}

Interpretation: a drifting mean or a variance ratio far from 1 signals non-stationarity; differencing removes a trend and a log transform stabilizes a growing variance. Edge cases: a moving average lags the series and is not a forecast; a centered versus trailing window answers different questions; a short series cannot support four chunks; seasonality is not removed by first differencing alone, so use a seasonal difference.

Mental model: a stationary series has a constant mean and variance, and a covariance that depends only on the lag. Trending and seasonal series are not stationary, and tests built for stationary data mislead on them.

How it works: first differencing removes a linear trend, a log transform stabilizes a growing variance, and a seasonal difference removes a repeating pattern. Formal checks are ADF, whose null is a unit root, and KPSS, whose null is stationarity.

Worked example: `[1, 2, 3, 4, 5, 6]` has a drifting mean and is non-stationary; `[3, 2, 4, 3, 3, 4]` is roughly stationary.

Common mistake: correlating two trending series and reading the high correlation as a relationship. That is spurious regression. Difference both series first, or model the trend explicitly.

## How do you bootstrap a confidence interval

The bootstrap resamples the observed data with replacement to approximate the sampling distribution of any statistic. It is the pragmatic answer for medians, ratios, and p99s, where a closed-form interval does not exist.

```text
values = [10, 12, 13, 15, 20, 900]; statistic = median
resample 10,000 times -> 95% CI around the median
```

In [25]:
try:
    import numpy as np

    def bootstrap_ci(values: list[float], draws: int = 10_000, alpha: float = 0.05, seed: int = 0) -> tuple[float, float]:
        rng = np.random.default_rng(seed)
        data = np.asarray(values)
        samples = np.array([np.median(rng.choice(data, size=data.size, replace=True)) for _ in range(draws)])
        return float(np.quantile(samples, alpha / 2)), float(np.quantile(samples, 1 - alpha / 2))
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - numpy is not installed; install it and re-run this cell.


Interpretation: the interval is only as good as the sample; bootstrap assumes the sample is representative and the observations are independent. Edge cases: resampling rows instead of users understates the interval when users have many rows each, so resample the cluster (the user); a heavy tail needs more draws for a stable p99; the percentile interval is biased for small samples, where a BCa interval is better; stratification should be preserved inside each resample.

Mental model: the sample stands in for the population, so resampling it with replacement mimics drawing from the population. The spread of the resampled statistic approximates its sampling distribution.

In [26]:
import random
import statistics

def bootstrap_ci(values: list[float], draws: int = 5_000, alpha: float = 0.05,
                 seed: int = 0) -> tuple[float, float]:
    rng = random.Random(seed)
    n = len(values)
    medians = sorted(statistics.median(rng.choices(values, k=n)) for _ in range(draws))
    low = medians[max(0, int(alpha / 2 * draws) - 1)]
    high = medians[min(draws - 1, int((1 - alpha / 2) * draws))]
    return low, high

low, high = bootstrap_ci([10, 12, 13, 15, 20, 900])
print(f"median = {statistics.median([10, 12, 13, 15, 20, 900])}, 95% CI = [{low}, {high}]")
assert low <= 14 <= high

median = 14.0, 95% CI = [11.0, 460.0]


How it works: draw n values with replacement, compute the statistic, repeat B times, and take the alpha/2 and 1 - alpha/2 quantiles. A few thousand draws are enough for a mean; a p99 needs far more.

Worked example: `[10, 12, 13, 15, 20, 900]` resampled 5,000 times with a fixed seed gives a 95% interval around the median that is far narrower than the range of the data.

Common mistake: resampling rows when the unit of analysis is the user, which understates the interval. Resample the cluster, and keep the seed so the interval is reproducible.

## How do you plan capacity with percentiles

Capacity planning uses peak demand plus headroom, not the average. Size from a high percentile of traffic, correct for latency at that load, and re-derive the numbers as the traffic mix changes.

```text
peak 12,000 rps, p95 6,000 rps, per replica 2,000 rps, headroom 1.3
replicas = ceil(12,000 * 1.3 / 2,000) = 8
```

In [27]:
import math

def replicas_for_peak(peak_rps: float, per_replica_rps: float, p99_ms: float, latency_budget_ms: float = 300.0, headroom: float = 1.3) -> dict[str, float]:
    # A replica that is slower than the budget serves proportionally less traffic.
    usable = per_replica_rps * min(1.0, latency_budget_ms / p99_ms)
    return {"usable_rps_per_replica": usable, "replicas": math.ceil(peak_rps * headroom / usable)}

Interpretation: size for the p95 or p99 of demand with explicit headroom, and treat the utilisation curve as non-linear — going from 70% to 90% utilisation more than doubles the queue. Edge cases: a daily peak is not a yearly peak, so use the seasonal maximum for provisioning; percentiles hide correlated bursts, so load-test the peak shape; autoscaling has a reaction delay, so the headroom must cover the scale-out time; a p99 that is latency-bound (for example a full garbage collection) does not scale linearly with replicas.

Mental model: the average is the wrong planning number because demand is bursty and queues grow non-linearly as utilization approaches one. Plan on a high percentile of demand plus explicit headroom.

How it works: `replicas = ceil(peak * headroom / usable_per_replica)`, and usable capacity per replica falls once its own p99 exceeds the latency budget. Little's law links the two: `L = lambda * W`, so queue length grows with arrival rate and residence time.

Worked example: peak 12,000 rps, 2,000 rps per replica, and a headroom factor of 1.3 give `ceil(15,600 / 2,000) = 8` replicas.

Common mistake: sizing on the daily average and then failing at the daily peak. Provisioning to 95% utilization is the second mistake: a small traffic bump crosses the knee of the queueing curve. Autoscaling also needs headroom to cover its reaction time.

## How do you implement statistics in SQL

Warehouses expose the same primitives as a stats library: `percentile_cont`, `stddev_samp`, `var_samp`, and histograms built with `width_bucket`. Push the aggregation into SQL so gigabytes do not move.

```sql
-- Percentiles and dispersion in one pass (PostgreSQL/Snowflake style)
select
  count(*)                                              as n,
  avg(latency_ms)                                       as mean,
  stddev_samp(latency_ms)                               as sd,
  var_samp(latency_ms)                                  as variance,
  percentile_cont(0.50) within group (order by latency_ms) as p50,
  percentile_cont(0.95) within group (order by latency_ms) as p95,
  percentile_cont(0.99) within group (order by latency_ms) as p99
from metrics.requests
where event_date between date '2026-09-01' and date '2026-09-30';
```

```sql
-- Histogram and a z-score based outlier flag
with buckets as (
  select width_bucket(latency_ms, 0, 1000, 20) as bucket, count(*) as requests
  from metrics.requests
  where event_date = current_date - 1
  group by 1
), stats as (
  select avg(latency_ms) as mean, stddev_samp(latency_ms) as sd from metrics.requests
)
select requests.event_id,
       (requests.latency_ms - stats.mean) / nullif(stats.sd, 0) as z_score
from metrics.requests cross join stats
where abs((requests.latency_ms - stats.mean) / nullif(stats.sd, 0)) > 3;
```

Interpretation: `percentile_cont` interpolates while `percentile_disc` returns an actual data point, so pick per the metric contract; `approx_percentile` trades a little accuracy for large speedups at scale. Edge cases: `stddev_samp` uses `n - 1` and `stddev_pop` uses `n`, and they differ most on small groups; percentiles cannot be averaged across partitions, so aggregate at the right grain; a z-score computed per partition must use the partition's own mean and standard deviation; NULLs are ignored by aggregates, which silently shrinks the denominator.

```sql
-- Rolling statistics with window functions; percentiles stay as grouped aggregates
with daily as (
  select event_date, service,
         avg(latency_ms)                                       as mean_ms,
         percentile_cont(0.95) within group (order by latency_ms) as p95_ms
  from metrics.requests
  group by event_date, service
)
select
  event_date,
  service,
  mean_ms,
  p95_ms,
  avg(mean_ms) over (partition by service order by event_date
                     rows between 6 preceding and current row) as mean_7d,
  count(*)     over (partition by service order by event_date
                     rows between 6 preceding and current row) as days_7d
from daily;
```

```sql
-- Nearest-rank percentile without percentile_cont
with ranked as (
  select service, latency_ms,
         percent_rank() over (partition by service order by latency_ms) as pr
  from metrics.requests
)
select service, min(latency_ms) as p95
from ranked
where pr >= 0.95
group by service;
```

`percent_rank` is the fraction of rows in the partition at or below the current row, so filtering at `pr >= 0.95` and taking the minimum reproduces a nearest-rank p95. The window form makes the definition explicit and lets the same query return a per-service percentile from one scan.

## How do you implement statistics in Python with numpy pandas and scipy

The Python stack splits cleanly: numpy for arrays and percentiles, pandas for grouped descriptive statistics, and scipy for tests and distributions.

In [28]:
try:
    import numpy as np
    import pandas as pd
    from scipy import stats

    latencies = np.array([10.0, 12.0, 13.0, 15.0, 20.0, 900.0])
    summary = {
        "mean": float(np.mean(latencies)),
        "std": float(np.std(latencies, ddof=1)),
        "p50": float(np.percentile(latencies, 50)),
        "p95": float(np.percentile(latencies, 95)),
        "p99": float(np.percentile(latencies, 99)),
        "skew": float(stats.skew(latencies)),
        "kurtosis": float(stats.kurtosis(latencies)),
    }

    frame = pd.DataFrame({"channel": ["a", "a", "b", "b"], "revenue": [10.0, 12.0, 40.0, 44.0]})
    by_channel = frame.groupby("channel")["revenue"].agg(["count", "mean", "std", "median"])

    test = stats.ttest_ind([10.0, 12.0, 13.0], [11.0, 14.0, 15.0], equal_var=False)
    interval = stats.norm.interval(0.95, loc=float(np.mean(latencies)), scale=float(stats.sem(latencies)))
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - numpy is not installed; install it and re-run this cell.


Interpretation: `np.std` defaults to the population denominator `ddof=0`, so pass `ddof=1` to match `statistics.stdev` and SQL's `stddev_samp`; `scipy.stats` covers tests, distributions, and intervals. Edge cases: `np.percentile` interpolates by default, so it can differ from nearest-rank SQL; pandas `agg(["mean", "std"])` skips NaN and can hide missing data; SciPy tests still assume independent observations; a pandas frame loaded with `toPandas()` on a large Spark result can exhaust driver memory.

In [29]:
import math

def percentile_cont(ordered: list[float], p: float) -> float:
    # Linear interpolation between the two closest ranks, as SQL percentile_cont does.
    if not ordered:
        raise ValueError("empty sample")
    position = (len(ordered) - 1) * p
    low, high = math.floor(position), math.ceil(position)
    if low == high:
        return ordered[low]
    return ordered[low] * (high - position) + ordered[high] * (position - low)

latencies = [10.0, 12.0, 13.0, 15.0, 20.0, 900.0]
print(f"p50 = {percentile_cont(latencies, 0.50)}, p95 = {percentile_cont(latencies, 0.95)}")
assert percentile_cont(latencies, 0.50) == 14.0

p50 = 14.0, p95 = 680.0


Standard-library equivalents: `statistics.stdev` matches `stddev_samp`, `math.erfc` gives a normal tail probability, and the `percentile_cont` above reproduces the interpolated SQL percentile. Reach for these when numpy or SciPy is not available, and keep the pandas and SciPy versions for grouped summaries and the full set of distributions.

## How do you apply statistics to data engineering decisions?

The sections so far explained the mechanics. This chapter turns them into four decisions a data engineer owns: where to set alert thresholds, how to experiment on the platform itself, how to size capacity against an objective, and how to read a statistic without fooling yourself.

## How do you set thresholds for data quality and anomaly alerts

Mental model: an alert is a classifier. The threshold trades false positives, which waste trust and cause people to ignore pages, against false negatives, which let real incidents through. The two costs are almost never symmetric.

How it works: learn a baseline from a robust statistic over a rolling window, set the cutoff from a high quantile of that baseline, and require a minimum effect size before paging. Median, MAD, and IQR survive the very outliers you are trying to detect, while mean and standard deviation are pulled by them.

In [30]:
import statistics

def alert_thresholds(history: list[float], robust_z: float = 3.5) -> dict[str, float]:
    median = statistics.median(history)
    mad = statistics.median([abs(value - median) for value in history]) or 1e-9
    return {"median": median, "mad": mad, "upper": median + robust_z * mad / 0.6745}

history = [100, 101, 99, 102, 98, 103, 100, 101, 100, 620]
bounds = alert_thresholds(history)
print(f"median = {bounds['median']}, threshold = {bounds['upper']:.1f}")
print("today 140 pages:", 140 > bounds["upper"])
assert round(bounds["upper"], 1) == 105.7
assert 140 > bounds["upper"] and 104 < bounds["upper"]

median = 100.5, threshold = 105.7
today 140 pages: True


Gotcha: a fixed percentage bound such as row count within 10% is wrong for any metric whose variance changes with day of week or volume, and it is either noisy or blind. Recompute thresholds from the recent distribution, and separate warning from page tiers. Measure the alert's own precision and recall on past incidents before shipping it.

## How do you run A B tests on data platform changes

Mental model: a platform change such as a new file format, a shuffling setting, or a compaction policy affects latency, cost, and reliability at once. A single is-it-faster number is not enough; decide the primary metric, the guardrails, and the randomization unit before flipping the switch.

How it works: randomize at a unit that cannot leak, such as a tenant or a cluster, rather than a query, because queries from one tenant share cache and shuffle. Compute the required sample from the minimum detectable effect, then check sample ratio mismatch before reading any result.

In [31]:
import math

def required_sample_per_arm(p1: float, p2: float, z_alpha: float = 1.959964,
                            z_power: float = 0.8416212) -> int:
    pooled = (p1 + p2) / 2
    numerator = (z_alpha * math.sqrt(2 * pooled * (1 - pooled))
                 + z_power * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return math.ceil(numerator / (p2 - p1) ** 2)

def srm_p_value(observed: list[int]) -> float:
    total = sum(observed)
    expected = [total / len(observed)] * len(observed)
    chi2 = sum((o - e) ** 2 / e for o, e in zip(observed, expected))
    return math.erfc(math.sqrt(chi2 / 2))

print("per arm for 10% to 12%:", required_sample_per_arm(0.10, 0.12))
print(f"srm p for [49100, 50900] = {srm_p_value([49100, 50900]):.2e}")
assert required_sample_per_arm(0.10, 0.12) == 3841
assert srm_p_value([49100, 50900]) < 0.001

per arm for 10% to 12%: 3841
srm p for [49100, 50900] = 1.25e-08


Gotcha: measuring the same workload before and after is a time-series comparison, not an experiment, because a weekday change or data growth will masquerade as an effect. Randomize concurrent cohorts instead. Treat a broken sample ratio mismatch as a stop sign: the cohorts are not comparable and no correction rescues the readout.

## How do you plan capacity and SLOs with percentiles

Mental model: a service-level objective is a statement about a percentile over a window, such as 99% of requests under 300 ms over 28 days. Capacity must serve the tail, so size on a high percentile of demand and leave headroom for the queueing knee.

How it works: p50 describes the typical request, p95 and p99 describe what users complain about, and the gap between them is the tail. `replicas = ceil(peak_demand * headroom / per_replica_capacity)`, where per-replica capacity falls once its own p99 exceeds the budget.

In [32]:
import math

def summarize(values: list[float], percentiles: tuple[float, ...] = (0.50, 0.95, 0.99)) -> dict[float, float]:
    ordered = sorted(values)
    out = {}
    for p in percentiles:
        position = (len(ordered) - 1) * p
        low, high = math.floor(position), math.ceil(position)
        out[p] = ordered[low] if low == high else ordered[low] * (high - position) + ordered[high] * (position - low)
    return out

latencies = [10, 12, 13, 15, 20, 40, 55, 90, 250, 900]
summary = summarize(latencies)
print({f"p{int(p * 100)}": round(v, 1) for p, v in summary.items()})
replicas = math.ceil(12_000 * 1.3 / 2_000)
print("replicas for 12,000 rps peak:", replicas)
assert round(summary[0.95], 1) == 607.5
assert round(summary[0.99], 1) == 841.5
assert replicas == 8

{'p50': 30.0, 'p95': 607.5, 'p99': 841.5}
replicas for 12,000 rps peak: 8


Gotcha: averaging per-hour percentiles to get a daily p99 is wrong because percentiles are not additive. Compute the percentile over the pooled raw events, or keep a sketch such as a t-digest per window. A p99 caused by a stop-the-world event does not divide across replicas, so adding capacity does not fix it.

## How do you read statistics responsibly

Mental model: every number carries a design. Before quoting one, ask which population it came from, what was measured, and what it was compared against.

How it works: correlation needs a mechanism and a design to become causal; multiple comparisons inflate false positives; survivorship bias hides the units that dropped out. These three are the most common ways a clean-looking number misleads.

In [33]:
import math

def two_sided_p_from_z(z: float) -> float:
    return math.erfc(abs(z) / math.sqrt(2))

def proportion_z(success_a: int, n_a: int, success_b: int, n_b: int) -> float:
    p_a, p_b = success_a / n_a, success_b / n_b
    pooled = (success_a + success_b) / (n_a + n_b)
    se = math.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    return (p_b - p_a) / se

# A 0.2 point lift is significant with a million users per arm and is still tiny.
z = proportion_z(100_000, 1_000_000, 102_000, 1_000_000)
print(f"z = {z:.2f}, p = {two_sided_p_from_z(z):.2e}, lift = 0.2 points")
assert two_sided_p_from_z(z) < 0.05
assert abs((102_000 / 1_000_000) - (100_000 / 1_000_000) - 0.002) < 1e-12

z = 4.69, p = 2.69e-06, lift = 0.2 points


Gotcha: significant means unlikely under the null, not large or important, and with enough data everything is significant. No significant difference means the test was underpowered, not that the effect is zero. Always pair an estimate with its uncertainty and with the decision it feeds.

## Statistics exercises

These exercises use only the Python standard library and compute the ideas on small, hard-coded samples. Each one states the task, a hint, and the expected result, then prints and asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The asserted numbers are a check on the formula, not a claim about real data. Real samples add variance, missing values, and cluster structure, so the value of each exercise is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Mean versus median on a skewed sample

**Task.** Given one skewed latency sample, compute the mean, the median, and the mean-to-median ratio, then classify the sample as heavy-tailed when the ratio exceeds 1.5.

**Hint.** `statistics.fmean` gives the mean and `statistics.median` gives the median; the ratio is the mean divided by the median.

**Expected result.** Mean 17.11, median 4, ratio 4.28, verdict `heavy-tailed`.

In [34]:
import statistics

latencies = [2, 3, 3, 4, 4, 5, 6, 7, 120]
mean = statistics.fmean(latencies)
median = statistics.median(latencies)
ratio = mean / median
verdict = "heavy-tailed" if ratio > 1.5 else "roughly symmetric"

print(f"mean = {mean:.2f}, median = {median:.2f}, mean/median = {ratio:.2f}")
print("verdict:", verdict)

assert round(mean, 2) == 17.11
assert median == 4
assert round(ratio, 2) == 4.28
assert verdict == "heavy-tailed"

mean = 17.11, median = 4.00, mean/median = 4.28
verdict: heavy-tailed


### Try it 2. Compute a percentile and explain p95

**Task.** Compute the nearest-rank p50 and p95 of the integers 1 to 20, then state in words what the p95 means.

**Hint.** Nearest rank is `ceil(pct / 100 * n)`; with 20 points the p95 rank is 19 and the p50 rank is 10.

**Expected result.** p50 = 10, p95 = 19, and 95% of requests are at or below the p95 value.

In [35]:
import math

def nearest_rank(ordered: list[float], pct: float) -> float:
    rank = max(1, math.ceil(pct / 100 * len(ordered)))
    return ordered[rank - 1]

values = list(range(1, 21))
p50 = nearest_rank(values, 50)
p95 = nearest_rank(values, 95)

print(f"p50 = {p50}, p95 = {p95}")
print("p95 means 95% of requests are at or below this value.")

assert p50 == 10
assert p95 == 19

p50 = 10, p95 = 19
p95 means 95% of requests are at or below this value.


### Try it 3. Build a confidence interval from mean and standard error

**Task.** From a sample mean of 20, a sample standard deviation of 10, and n = 100, compute the standard error and the 95% confidence interval.

**Hint.** `SE = stdev / sqrt(n)` and the interval is `mean +/- 1.96 * SE` for a large sample.

**Expected result.** SE = 1.0 and the interval is [18.04, 21.96].

In [36]:
import math

mean = 20.0
stdev = 10.0
n = 100
z = 1.959964

standard_error = stdev / math.sqrt(n)
low, high = mean - z * standard_error, mean + z * standard_error

print(f"SE = {standard_error:.2f}, 95% CI = [{low:.2f}, {high:.2f}]")
# The interval is about the procedure: 95% of intervals built this way cover the true mean.

assert round(standard_error, 2) == 1.0
assert round(low, 2) == 18.04
assert round(high, 2) == 21.96

SE = 1.00, 95% CI = [18.04, 21.96]


### Try it 4. Compute required sample size from an effect size

**Task.** For a baseline conversion of 10% and a target of 12%, compute the required sample per arm at alpha 0.05 and 80% power.

**Hint.** Use the pooled and unpooled variance terms: `(z_alpha * sqrt(2 p(1-p)) + z_power * sqrt(p1(1-p1) + p2(1-p2)))^2 / (p2 - p1)^2`, then round up.

**Expected result.** About 3,841 users per arm.

In [37]:
import math

def sample_size_per_arm(p1: float, p2: float, z_alpha: float = 1.959964,
                        z_power: float = 0.8416212) -> int:
    pooled = (p1 + p2) / 2
    numerator = (z_alpha * math.sqrt(2 * pooled * (1 - pooled))
                 + z_power * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return math.ceil(numerator / (p2 - p1) ** 2)

n = sample_size_per_arm(0.10, 0.12)
print("required per arm:", n)

assert n == 3841

required per arm: 3841


### Try it 5. Apply Bonferroni and Benjamini-Hochberg

**Task.** Given ten p-values, count how many hypotheses Bonferroni rejects at alpha 0.05 and how many Benjamini-Hochberg rejects.

**Hint.** Bonferroni uses the cutoff `alpha / m`. Benjamini-Hochberg sorts the p-values and rejects the largest rank k where `p_(k) <= k / m * alpha`.

**Expected result.** Bonferroni rejects 2 and Benjamini-Hochberg rejects 4.

In [38]:
p_values = [0.001, 0.004, 0.01, 0.02, 0.03, 0.2, 0.5, 0.7, 0.9, 0.95]
alpha = 0.05
m = len(p_values)

bonferroni = [i for i, p in enumerate(p_values) if p <= alpha / m]

ranked = sorted(enumerate(p_values), key=lambda item: item[1])
cutoff_rank = 0
for rank, (_index, p) in enumerate(ranked, start=1):
    if p <= rank / m * alpha:
        cutoff_rank = rank
benjamini_hochberg = [i for i, _p in ranked[:cutoff_rank]]

print("bonferroni rejects:", len(bonferroni))
print("benjamini-hochberg rejects:", len(benjamini_hochberg))

assert len(bonferroni) == 2
assert len(benjamini_hochberg) == 4

bonferroni rejects: 2
benjamini-hochberg rejects: 4


### Try it 6. Detect sample ratio mismatch

**Task.** An experiment intended 50/50 observed 49,100 control and 50,900 treatment. Compute the chi-square statistic and its p-value and decide whether there is sample ratio mismatch at alpha 0.001.

**Hint.** For a two-arm split, `chi2 = sum((observed - expected)^2 / expected)` and, with one degree of freedom, `p = erfc(sqrt(chi2 / 2))`.

**Expected result.** chi2 = 32.4, p about 1.3e-08, so there is sample ratio mismatch.

In [39]:
import math

control, treatment = 49_100, 50_900
total = control + treatment
expected = [total / 2, total / 2]
observed = [control, treatment]

chi2 = sum((o - e) ** 2 / e for o, e in zip(observed, expected))
p_value = math.erfc(math.sqrt(chi2 / 2))
srm = p_value < 0.001

print(f"chi2 = {chi2:.1f}, p = {p_value:.2e}, srm = {srm}")

assert round(chi2, 1) == 32.4
assert srm is True

chi2 = 32.4, p = 1.25e-08, srm = True


### Try it 7. Compare z-score and MAD thresholds on an outlier

**Task.** For `[10, 11, 12, 12, 13, 900]`, compute the plain z-score of the last value and the robust z-score from the median and MAD. Report which method flags it at their usual thresholds of 3.0 and 3.5.

**Hint.** The robust z-score is `0.6745 * (x - median) / MAD`; the plain z-score is `(x - mean) / stdev`.

**Expected result.** The plain z-score is about 2.04 and does not flag, while the robust z-score is about 598.96 and does flag.

In [40]:
import statistics

values = [10, 11, 12, 12, 13, 900]
mean = statistics.fmean(values)
stdev = statistics.stdev(values)
z_score = abs(900 - mean) / stdev

median = statistics.median(values)
mad = statistics.median([abs(v - median) for v in values])
robust_z = 0.6745 * abs(900 - median) / mad

print(f"z-score = {z_score:.2f} (flags: {z_score > 3.0})")
print(f"robust z = {robust_z:.2f} (flags: {robust_z > 3.5})")

assert round(z_score, 2) == 2.04
assert z_score < 3.0
assert robust_z > 3.5

z-score = 2.04 (flags: False)
robust z = 598.96 (flags: True)


### Try it 8. Bootstrap a confidence interval

**Task.** Bootstrap a 95% confidence interval for the median of `[10, 12, 13, 15, 20, 900]` with 5,000 resamples and a fixed seed.

**Hint.** Resample with replacement at the original size, take the median of each resample, and read the 2.5th and 97.5th percentiles. Use `random.Random(seed)` so the answer is reproducible.

**Expected result.** The median is 14 and the interval is [11, 460], which contains the median and stays below the maximum.

In [41]:
import random
import statistics

def bootstrap_median_ci(values: list[float], draws: int = 5_000, alpha: float = 0.05,
                        seed: int = 0) -> tuple[float, float]:
    rng = random.Random(seed)
    n = len(values)
    medians = sorted(statistics.median(rng.choices(values, k=n)) for _ in range(draws))
    low = medians[max(0, int(alpha / 2 * draws) - 1)]
    high = medians[min(draws - 1, int((1 - alpha / 2) * draws))]
    return low, high

values = [10, 12, 13, 15, 20, 900]
low, high = bootstrap_median_ci(values)

print(f"median = {statistics.median(values)}, 95% CI = [{low}, {high}]")

assert low <= statistics.median(values) <= high
assert high < max(values)

median = 14.0, 95% CI = [11.0, 460.0]


## Real world use cases for statistics in data engineering

### 1. Experimentation platform for a consumer app

The platform assigns users by hashing, checks for sample ratio mismatch on every dimension, and reports lift with a confidence interval and a guardrail dashboard. It blocks a launch when any pre-registered guardrail regresses.

In [42]:
try:
    from scipy.stats import chisquare

    counts = {"control": 49_100, "treatment": 50_900}
    expected = [50_000, 50_000]
    chi2, p_value = chisquare(list(counts.values()), expected)
    assert p_value < 0.001  # SRM: the readout is not trustworthy
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - scipy is not installed; install it and re-run this cell.


Why it matters: an automated SRM gate catches broken assignment and logging before a false result reaches a decision maker.

### 2. Data-quality thresholds derived from distributions

Instead of hard-coding "row count must be within 10%", the pipeline learns the distribution of each check over 90 days and alerts when today's value falls outside the 1st or 99th percentile.

In [43]:
try:
    import numpy as np

    def threshold_from_history(history: list[float], alpha: float = 0.01) -> tuple[float, float]:
        return float(np.quantile(history, alpha / 2)), float(np.quantile(history, 1 - alpha / 2))
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - numpy is not installed; install it and re-run this cell.


Why it matters: distribution-based thresholds adapt to weekday and seasonal patterns and cut alert fatigue compared with fixed bounds.

### 3. Capacity planning for a streaming cluster

The team plots rps and consumer lag by percentile, models peak-plus-headroom, and reprovisions before a marketing event instead of after an incident.

In [44]:
import math

def brokers_needed(peak_records_per_sec: float, records_per_broker: float, headroom: float = 1.4) -> int:
    return math.ceil(peak_records_per_sec * headroom / records_per_broker)

Why it matters: peak demand plus explicit headroom is defensible, whereas sizing on the average guarantees saturation during the daily peak.

### 4. Anomaly alerting on pipeline metrics

Freshness, null rate, and schema-drift counts are monitored with robust z-scores on a rolling baseline, so a single extreme day does not desensitize the detector.

In [45]:
import statistics

def robust_flag(value: float, history: list[float], threshold: float = 3.5) -> bool:
    median = statistics.median(history)
    mad = statistics.median([abs(item - median) for item in history]) or 1e-9
    return 0.6745 * abs(value - median) / mad > threshold

Why it matters: median and MAD are unaffected by past outliers, which is exactly what an alerting baseline needs.

### 5. Marketing attribution and incrementality

Correlation between spend and revenue is not incremental revenue. The team runs geo holdouts and computes the lift with a difference-in-differences estimate instead of a raw spend correlation.

In [46]:
def difference_in_differences(treat_pre: float, treat_post: float, control_pre: float, control_post: float) -> float:
    return (treat_post - treat_pre) - (control_post - control_pre)

Why it matters: a holdout removes the confounding that makes spend always look effective, and the estimator stays explainable to finance.

### 6. Fraud detection thresholds

The model scores transactions and the threshold is chosen from the precision-recall curve at the operating point that controls expected loss, not from accuracy on an imbalanced dataset.

In [47]:
try:
    from sklearn.metrics import precision_recall_curve

    def threshold_at_recall(y_true, scores, target_recall: float = 0.8) -> float:
        precision, recall, thresholds = precision_recall_curve(y_true, scores)
        eligible = [thresholds[i] for i in range(len(thresholds)) if recall[i] >= target_recall]
        return float(min(eligible)) if eligible else float(thresholds[-1])
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")


SKIPPED - sklearn is not installed; install it and re-run this cell.


Why it matters: fraud is rare, so a percentage accuracy of 99.9% is meaningless; precision and recall at a business-chosen operating point is the honest metric.